In [ ]:
# Upload datasets as .csv files in your colab session, and then run this cell.
# Dataset Link: https://drive.google.com/drive/folders/1PzURxyRaN5rclRcyRSI9G8oa1XVSo_SF?usp=drive_link

# Don't work in this colab. Make a copy in your drive instead.

import pandas as pd

copd_df = '/content/COPD_data_preprocessed.csv'
alzheimers_df = '/content/alzheimers_disease_data_preprocessed.csv'
arrhythmia_df = '/content/arrhythmia_data_preprocessed.csv'
autism_df = '/content/autism_screening_data_preprocessed.csv'
diabetes_df = '/content/diabetes_data_preprocessed.csv'
stroke_df = '/content/healthcare-dataset-stroke-data_preprocessed.csv'
heart_df = '/content/heart_disease_uci_preprocessed.csv'
hepatitis_df = '/content/hepatitis_data_preprocessed.csv'
liver_df = '/content/liver_patient_dataset_preprocessed.csv'
thyroid_df = '/content/thyroid_dataset_preprocessed.csv'

In [ ]:
# Cell 2: Install TabFM v1.0.0 (PyTorch backend) and required packages
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # Prevents HF Xet transfer stalls in Colab

!pip install -q "tabfm[pytorch] @ git+https://github.com/google-research/tabfm" safetensors statsmodels

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
inflect 7.6.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [ ]:
# Cell 3: Setup, Configuration, and Load Pretrained TabFM v1.0.0
import gc
import os
import warnings
import numpy as np
import pandas as pd
import torch
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning

from sklearn.model_selection import StratifiedKFold
from sklearn.impute import KNNImputer
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

# ==========================================
# Global Benchmark Configuration
# ==========================================
RANDOM_STATE = 42       # Keep identical across all 4 researchers/models
N_SPLITS = 5           # 5-fold stratified cross-validation
THRESHOLD = 0.5         # Fixed classification threshold
KNN_NEIGHBORS = 5       # KNN imputation neighbors inside each fold
NOISE_FRACTION = 0.10   # 10% of training-set feature SD for robustness noise

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
print(f"Using device: {DEVICE} | dtype: {DTYPE}")

# Load TabFM v1.0.0 classification weights once (downloads automatically from HF)
base_tabfm_model = tabfm_v1_0_0.load(
    model_type="classification",
    device=DEVICE,
    dtype=DTYPE,
)
print("TabFM v1.0.0 classification weights loaded successfully.")

# Map the 10 variables defined in Cell 1 -> (Dataset Identifier, Variable/Path, Healthcare Domain)
DATASETS = [
    ("COPD", copd_df, "Pulmonology / Respiratory"),
    ("Alzheimers", alzheimers_df, "Neurology"),
    ("Arrhythmia", arrhythmia_df, "Cardiology"),
    ("Autism", autism_df, "Neurodevelopmental / Psychiatry"),
    ("Diabetes", diabetes_df, "Endocrinology"),
    ("Stroke", stroke_df, "Neurology / Cerebrovascular"),
    ("Heart_Disease", heart_df, "Cardiology"),
    ("Hepatitis", hepatitis_df, "Hepatology / Infectious Disease"),
    ("Liver_Disease", liver_df, "Hepatology"),
    ("Thyroid", thyroid_df, "Endocrinology"),
]

Using device: cuda | dtype: torch.bfloat16


config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights from local directory
TabFM v1.0.0 classification weights loaded successfully.


In [ ]:
# Cell 4: Exact Metric Implementations (Protocol Section 3 & 4)

def _safe_div(num, den):
    return float(num / den) if den > 0 else np.nan


def compute_fairness_metrics(y_true, y_pred, sex):
    """
    Protected groups: Female (sex == 0), Male (sex == 1)
    Returns DI, EOD, DPD, EqOD, PPD (with N/A -> np.nan on zero denominators).
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    sex = np.asarray(sex, dtype=int)

    stats = {}
    for group_name, group_val in [("Male", 1), ("Female", 0)]:
        mask = (sex == group_val)
        n_g = np.sum(mask)
        if n_g == 0:
            stats[group_name] = {"SR": np.nan, "TPR": np.nan, "FPR": np.nan, "PPV": np.nan}
            continue

        yt = y_true[mask]
        yp = y_pred[mask]

        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))

        sr = float(np.mean(yp == 1))
        tpr = _safe_div(tp, tp + fn)
        fpr = _safe_div(fp, fp + tn)
        ppv = _safe_div(tp, tp + fp)

        stats[group_name] = {"SR": sr, "TPR": tpr, "FPR": fpr, "PPV": ppv}

    sr_m, sr_f = stats["Male"]["SR"], stats["Female"]["SR"]
    tpr_m, tpr_f = stats["Male"]["TPR"], stats["Female"]["TPR"]
    fpr_m, fpr_f = stats["Male"]["FPR"], stats["Female"]["FPR"]
    ppv_m, ppv_f = stats["Male"]["PPV"], stats["Female"]["PPV"]

    # Disparate Impact (DI): N/A if both selection rates are zero or either group missing
    if np.isnan(sr_m) or np.isnan(sr_f) or max(sr_m, sr_f) == 0.0:
        di = np.nan
    else:
        di = float(min(sr_m, sr_f) / max(sr_m, sr_f))

    # Equal Opportunity Difference (EOD)
    eod = np.nan if (np.isnan(tpr_m) or np.isnan(tpr_f)) else float(abs(tpr_m - tpr_f))

    # Demographic Parity Difference (DPD)
    dpd = np.nan if (np.isnan(sr_m) or np.isnan(sr_f)) else float(abs(sr_m - sr_f))

    # Equalized Odds Difference (EqOD)
    if any(np.isnan(v) for v in [tpr_m, tpr_f, fpr_m, fpr_f]):
        eqod = np.nan
    else:
        eqod = float(max(abs(tpr_m - tpr_f), abs(fpr_m - fpr_f)))

    # Predictive Parity Difference (PPD)
    ppd = np.nan if (np.isnan(ppv_m) or np.isnan(ppv_f)) else float(abs(ppv_m - ppv_f))

    return {
        "DI": di,
        "EOD": eod,
        "DPD": dpd,
        "EqOD": eqod,
        "PPD": ppd,
    }


def compute_discrimination_metrics(y_true, y_pred, y_prob):
    """
    Returns AUROC, Sensitivity, Specificity, PPV, NPV.
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    auroc = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan
    sensitivity = _safe_div(tp, tp + fn)
    specificity = _safe_div(tn, tn + fp)
    ppv = _safe_div(tp, tp + fp)
    npv = _safe_div(tn, tn + fn)

    return {
        "AUROC": auroc,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "PPV": ppv,
        "NPV": npv,
    }


def compute_calibration_metrics(y_true, y_prob):
    """
    Returns ECE (15 equal-width bins), NLL, Brier, Cal_Slope, Cal_Intercept.
    Probabilities clipped to [1e-15, 1 - 1e-15] before ECE/NLL/logit.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    p_raw = np.asarray(y_prob, dtype=np.float64)
    p_clip = np.clip(p_raw, 1e-15, 1.0 - 1e-15)

    # 1. ECE: 15 equal-width bins covering [0, 1], last bin includes 1.0
    bin_edges = np.linspace(0.0, 1.0, 16)
    n_total = len(y_true)
    ece = 0.0
    for b in range(15):
        if b < 14:
            mask = (p_clip >= bin_edges[b]) & (p_clip < bin_edges[b + 1])
        else:
            mask = (p_clip >= bin_edges[b]) & (p_clip <= bin_edges[b + 1])
        n_b = np.sum(mask)
        if n_b > 0:
            acc_b = np.mean(y_true[mask])
            conf_b = np.mean(p_clip[mask])
            ece += (n_b / n_total) * abs(acc_b - conf_b)

    # 2. NLL
    nll = -np.mean(y_true * np.log(p_clip) + (1.0 - y_true) * np.log(1.0 - p_clip))

    # 3. Brier Score
    brier = np.mean((p_raw - y_true) ** 2)

    # 4-5. Logistic Calibration Regression: logit(P(Y=1)) = a + b * logit(p)
    cal_intercept = np.nan
    cal_slope = np.nan
    logit_p = np.log(p_clip / (1.0 - p_clip))

    if len(np.unique(y_true)) == 2 and np.std(logit_p) > 0:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                X_design = sm.add_constant(logit_p, has_constant="add")
                fit_res = sm.Logit(y_true, X_design).fit(disp=False, maxiter=1000)
                if fit_res.mle_retvals.get("converged", False):
                    cal_intercept = float(fit_res.params[0])
                    cal_slope = float(fit_res.params[1])
        except Exception:
            cal_intercept = np.nan
            cal_slope = np.nan

    return {
        "ECE": float(ece),
        "NLL": float(nll),
        "Brier": float(brier),
        "Cal_Slope": cal_slope,
        "Cal_Intercept": cal_intercept,
    }


def compute_robustness_metrics(y_true, y_pred_clean, p_clean, y_pred_noisy, p_noisy):
    """
    Returns AUROC_Deg, Accuracy_Deg, F1_Deg, Flip_Rate, MAPC.
    Degradation = M_clean - M_noisy (keep signed values; do not clamp or abs).
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred_clean = np.asarray(y_pred_clean, dtype=int)
    y_pred_noisy = np.asarray(y_pred_noisy, dtype=int)
    p_clean = np.asarray(p_clean, dtype=np.float64)
    p_noisy = np.asarray(p_noisy, dtype=np.float64)

    if len(np.unique(y_true)) == 2:
        auroc_clean = roc_auc_score(y_true, p_clean)
        auroc_noisy = roc_auc_score(y_true, p_noisy)
        auroc_deg = float(auroc_clean - auroc_noisy)
    else:
        auroc_deg = np.nan

    acc_clean = accuracy_score(y_true, y_pred_clean)
    acc_noisy = accuracy_score(y_true, y_pred_noisy)
    acc_deg = float(acc_clean - acc_noisy)

    f1_clean = f1_score(y_true, y_pred_clean, pos_label=1, zero_division=0.0)
    f1_noisy = f1_score(y_true, y_pred_noisy, pos_label=1, zero_division=0.0)
    f1_deg = float(f1_clean - f1_noisy)

    flip_rate = float(np.mean(y_pred_clean != y_pred_noisy))
    mapc = float(np.mean(np.abs(p_clean - p_noisy)))

    return {
        "AUROC_Deg": auroc_deg,
        "Accuracy_Deg": acc_deg,
        "F1_Deg": f1_deg,
        "Flip_Rate": flip_rate,
        "MAPC": mapc,
    }


def summarize_fold_values(values):
    """
    Computes arithmetic mean and sample SD (ddof=1), ignoring N/A (NaN) folds.
    Returns (np.nan, np.nan) if no valid folds exist, and sd=np.nan if <2 valid folds.
    """
    arr = np.asarray(values, dtype=np.float64)
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return np.nan, np.nan
    mean_val = float(np.mean(valid))
    sd_val = float(np.std(valid, ddof=1)) if len(valid) >= 2 else np.nan
    return mean_val, sd_val

In [ ]:
# Cell 5: Run TabFM v1.0.0 Evaluation Across All 10 Datasets (with single tqdm progress bar)
from tqdm.auto import tqdm

CSV_COLUMNS = [
    "Dataset", "Train_N", "Test_N", "Num_Feats", "Domain",
    "DI_mean", "DI_sd",
    "EOD_mean", "EOD_sd",
    "DPD_mean", "DPD_sd",
    "EqOD_mean", "EqOD_sd",
    "PPD_mean", "PPD_sd",
    "AUROC_mean", "AUROC_sd",
    "Sensitivity_mean", "Sensitivity_sd",
    "Specificity_mean", "Specificity_sd",
    "PPV_mean", "PPV_sd",
    "NPV_mean", "NPV_sd",
    "ECE_mean", "ECE_sd",
    "NLL_mean", "NLL_sd",
    "Brier_mean", "Brier_sd",
    "Cal_Slope_mean", "Cal_Slope_sd",
    "Cal_Intercept_mean", "Cal_Intercept_sd",
    "AUROC_Deg_mean", "AUROC_Deg_sd",
    "Accuracy_Deg_mean", "Accuracy_Deg_sd",
    "F1_Deg_mean", "F1_Deg_sd",
    "Flip_Rate_mean", "Flip_Rate_sd",
    "MAPC_mean", "MAPC_sd",
]

METRIC_KEYS = [
    "DI", "EOD", "DPD", "EqOD", "PPD",
    "AUROC", "Sensitivity", "Specificity", "PPV", "NPV",
    "ECE", "NLL", "Brier", "Cal_Slope", "Cal_Intercept",
    "AUROC_Deg", "Accuracy_Deg", "F1_Deg", "Flip_Rate", "MAPC",
]

all_results = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

total_folds = len(DATASETS) * N_SPLITS  # 10 datasets * 10 folds = 100 (1 fold = 1%)

with tqdm(total=total_folds, desc="TabFM Benchmark", unit="fold") as pbar:
    for ds_name, ds_source, ds_domain in DATASETS:
        df = pd.read_csv(ds_source) if isinstance(ds_source, str) else ds_source.copy()

        # Validate column positions: first column = "sex", last column = "outcome"
        sex_col = df.columns[0]
        target_col = df.columns[-1]
        feature_cols = list(df.columns[:-1])

        if sex_col.lower() != "sex":
            tqdm.write(f"[Warning] {ds_name}: First column is '{sex_col}', expected 'sex'.")
        if target_col.lower() != "outcome":
            tqdm.write(f"[Warning] {ds_name}: Last column is '{target_col}', expected 'outcome'.")

        X = df[feature_cols].copy()
        y = df[target_col].astype(int).copy()

        fold_metrics = {k: [] for k in METRIC_KEYS}
        train_sizes = []
        test_sizes = []

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
            pbar.set_description(f"TabFM [{ds_name} | Fold {fold_idx}/{N_SPLITS}]")

            X_train_raw = X.iloc[train_idx].copy()
            X_test_raw = X.iloc[test_idx].copy()
            y_train = y.iloc[train_idx].to_numpy(dtype=int)
            y_test = y.iloc[test_idx].to_numpy(dtype=int)

            train_sizes.append(len(train_idx))
            test_sizes.append(len(test_idx))

            # 1. KNN Imputation strictly inside the fold (fit on train, transform train & test)
            imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
            X_train_imp = pd.DataFrame(
                imputer.fit_transform(X_train_raw),
                columns=feature_cols,
                index=X_train_raw.index,
            )
            X_test_clean = pd.DataFrame(
                imputer.transform(X_test_raw),
                columns=feature_cols,
                index=X_test_raw.index,
            )

            # Preserve binary integer encoding for "sex"
            X_train_imp[sex_col] = np.round(X_train_imp[sex_col]).astype(int)
            X_test_clean[sex_col] = np.round(X_test_clean[sex_col]).astype(int)
            sex_test = X_test_clean[sex_col].to_numpy(dtype=int)

            # 2. Fit TabFM v1.0.0 (sets context from training fold) & predict clean test fold
            clf = TabFMClassifier(model=base_tabfm_model, random_state=RANDOM_STATE)
            clf.fit(X_train_imp, y_train)

            pos_idx = list(clf.classes_).index(1)
            probs_clean = clf.predict_proba(X_test_clean)
            p_clean = np.asarray(probs_clean[:, pos_idx], dtype=np.float64)
            y_pred_clean = (p_clean >= THRESHOLD).astype(int)

            # 3. Noise injection on test features (all features except "sex")
            X_test_noisy = X_test_clean.copy()
            for col in feature_cols:
                if col == sex_col:
                    continue  # Never perturb "sex"
                sd_train = float(X_train_imp[col].std(ddof=1))  # Sample SD of training feature
                if np.isnan(sd_train) or sd_train == 0.0:
                    continue
                noise_scale = NOISE_FRACTION * sd_train
                eps = rng.normal(loc=0.0, scale=1.0, size=len(X_test_noisy))
                X_test_noisy[col] = X_test_noisy[col] + noise_scale * eps

            probs_noisy = clf.predict_proba(X_test_noisy)
            p_noisy = np.asarray(probs_noisy[:, pos_idx], dtype=np.float64)
            y_pred_noisy = (p_noisy >= THRESHOLD).astype(int)

            # 4. Compute all 20 fold-level metrics
            fold_res = {}
            fold_res.update(compute_fairness_metrics(y_test, y_pred_clean, sex_test))
            fold_res.update(compute_discrimination_metrics(y_test, y_pred_clean, p_clean))
            fold_res.update(compute_calibration_metrics(y_test, p_clean))
            fold_res.update(compute_robustness_metrics(y_test, y_pred_clean, p_clean, y_pred_noisy, p_noisy))

            for k in METRIC_KEYS:
                fold_metrics[k].append(fold_res[k])

            # Update single progress bar
            pbar.set_postfix(
                AUROC=f"{fold_res['AUROC']:.4f}",
                ECE=f"{fold_res['ECE']:.4f}",
                Flip=f"{fold_res['Flip_Rate']:.4f}",
            )
            pbar.update(1)

            # Free temporary GPU memory per fold
            del clf
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        # Aggregate dataset row (no intermediate rounding; ddof=1 for SD)
        avg_train_n = float(np.mean(train_sizes))
        avg_test_n = float(np.mean(test_sizes))
        row = {
            "Dataset": ds_name,
            "Train_N": int(avg_train_n) if avg_train_n.is_integer() else avg_train_n,
            "Test_N": int(avg_test_n) if avg_test_n.is_integer() else avg_test_n,
            "Num_Feats": len(feature_cols),
            "Domain": ds_domain,
        }
        for k in METRIC_KEYS:
            m, s = summarize_fold_values(fold_metrics[k])
            row[f"{k}_mean"] = m
            row[f"{k}_sd"] = s

        all_results.append(row)
        gc.collect()

TabFM Benchmark:   0%|          | 0/50 [00:00<?, ?fold/s]

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
/usr/local/lib/python3.13/dist-packages/statsmodels/base/optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may

In [ ]:
# Cell 6: Save TabFM_Results.csv in Exact Protocol Column Order & Download
results_df = pd.DataFrame(all_results)[CSV_COLUMNS]

output_filename = "TabFM_Results.csv"
results_df.to_csv(output_filename, index=False, na_rep="N/A")

print(f"Saved results to {output_filename} with shape {results_df.shape}")
display(results_df)

# Optional: automatically download the CSV in Google Colab
try:
    from google.colab import files
    files.download(output_filename)
except ImportError:
    pass

Saved results to TabFM_Results.csv with shape (10, 45)


,Dataset,Train_N,Test_N,Num_Feats,Domain,DI_mean,DI_sd,EOD_mean,EOD_sd,DPD_mean,...,AUROC_Deg_mean,AUROC_Deg_sd,Accuracy_Deg_mean,Accuracy_Deg_sd,F1_Deg_mean,F1_Deg_sd,Flip_Rate_mean,Flip_Rate_sd,MAPC_mean,MAPC_sd
0,COPD,504.0,126.0,14,Pulmonology / Respiratory,0.631603,0.149963,0.100805,0.160176,0.312810,...,0.021657,0.024729,0.011111,0.034412,0.012791,0.024955,0.112698,0.028835,0.200082,0.018617
1,Alzheimers,560.0,140.0,32,Neurology,0.827423,0.143301,0.044671,0.034399,0.098230,...,0.015980,0.007364,0.124286,0.034479,0.105461,0.027714,0.127143,0.030888,0.247439,0.020984
2,Arrhythmia,361.6,90.4,262,Cardiology,0.549409,0.143553,0.129981,0.082272,0.247202,...,0.007464,0.009124,0.332088,0.061580,0.152263,0.062303,0.548742,0.080263,0.312487,0.035383
3,Autism,559.2,139.8,19,Neurodevelopmental / Psychiatry,0.711783,0.139244,0.000000,0.000000,0.094228,...,0.000000,0.000000,0.008592,0.006001,0.015721,0.011029,0.008592,0.006001,0.196988,0.017369
4,Diabetes,560.0,140.0,8,Endocrinology,0.882050,0.095711,0.065606,0.047597,0.064226,...,0.016980,0.009299,0.038571,0.024011,0.028587,0.023865,0.101429,0.024432,0.136288,0.005065
5,Stroke,560.0,140.0,10,Neurology / Cerebrovascular,0.924368,0.079119,0.127486,0.058759,0.026251,...,0.002159,0.003539,0.007143,0.020825,-0.011013,0.025420,0.044286,0.009313,0.083498,0.013846
6,Heart_Disease,560.0,140.0,14,Cardiology,0.277115,0.026203,0.196247,0.142434,0.442612,...,0.039429,0.015125,0.048571,0.038598,0.034758,0.024087,0.122857,0.058117,0.209189,0.013788
7,Hepatitis,124.0,31.0,19,Hepatology / Infectious Disease,0.134615,0.301009,NaN,NaN,0.188912,...,0.013571,0.055353,0.000000,0.060349,-0.002009,0.069566,0.064516,0.060349,0.141817,0.026141
8,Liver_Disease,560.0,140.0,10,Hepatology,0.879827,0.074564,0.039623,0.045584,0.071742,...,0.238265,0.008686,0.388571,0.021783,0.237900,0.015986,0.434286,0.043272,0.344771,0.025248
9,Thyroid,560.0,140.0,21,Endocrinology,0.771783,0.124540,0.003509,0.007846,0.126813,...,0.156980,0.031035,0.191429,0.015485,0.204346,0.015011,0.205714,0.011737,0.315889,0.023554


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>